# C7-cnn-transfer — Practice p13 — Solution

**(a) Update claim.** One output at layer $\ell$ reads $K_\ell$
consecutive positions from layer $\ell-1$. The first such position sees an
interval of $r_{\ell-1}$ input pixels. Each later position starts
$J_{\ell-1}$ input pixels after the preceding one, so the final position
extends the union by $(K_\ell-1)J_{\ell-1}$ pixels. Convolutional windows
overlap when $J_{\ell-1}<r_{\ell-1}$ and tile at equality; in either case
their union is one interval with length
$r_\ell=r_{\ell-1}+(K_\ell-1)J_{\ell-1}$.

**Assumption needed.** The overlap-or-tile sentence requires
$J_{\ell-1}\le r_{\ell-1}$, as in the course's stacks and the anchor below.
If arbitrary strides may skip farther than the preceding receptive field spans,
the sampled support can have holes; the same recurrence then measures its
bounding span, not the cardinality of a contiguous union. The prompt does not
state this no-gap assumption explicitly.

The current layer's stride changes the spacing *between its outputs*, not the
locations read inside the one output whose field is being computed. Therefore
the receptive-field update uses the old jump first, and only afterward does
$J_\ell=J_{\ell-1}s_\ell$.

**(b) Closed-form claim.** Starting from $r_0=1$ and substituting
$J_{\ell-1}=\prod_{i<\ell}s_i$ into each update, telescoping gives
$$r_L=1+\sum_{\ell=1}^L (K_\ell-1)\prod_{i<\ell}s_i.$$
When every stride is one, each product is one, hence
$r_L=1+\sum_{\ell=1}^L(K_\ell-1)$.


In [ ]:
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)   # course convention (no pretrained weights here)
SEED = 20260804
torch.manual_seed(SEED)

def ones_conv1d(K, s):
    layer = nn.Conv1d(1, 1, kernel_size=K, stride=s, bias=False)
    layer.weight = nn.Parameter(torch.ones(1, 1, K), requires_grad=False)
    return layer

stack = nn.Sequential(ones_conv1d(3, 1), ones_conv1d(5, 2), ones_conv1d(3, 1))
N = 101
base = torch.zeros(1, 1, N)
with torch.inference_mode():
    out0 = stack(base)
center = out0.shape[-1] // 2

# 1 + (3-1)*1 + (5-1)*1 + (3-1)*(1*2) = 11.
rf_formula = 11
rf_measured = 0
with torch.inference_mode():
    for i in range(N):
        probe = base.clone()
        probe[0, 0, i] = 1.0
        rf_measured += int(stack(probe)[0, 0, center] != out0[0, 0, center])
anchor_gap = abs(rf_formula - rf_measured)


### Answer check

In [ ]:
assert rf_formula == 11
assert rf_measured == 11
assert anchor_gap == 0
